In [1]:
import ast
import numpy as np
import pandas as pd
import scipy.sparse as sp
from datetime import datetime, timedelta
import itertools

from sklearn import logger

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark.ml.recommendation import ALS
from pyspark.mllib.evaluation import RankingMetrics

from lightfm.data import Dataset
from lightfm import LightFM
from lightfm.evaluation import precision_at_k, recall_at_k

import mlflow
import mlflow.spark

import mlflow

In [2]:
spark = SparkSession.builder \
    .appName("als_testbench") \
    .config("spark.sql.shuffle.partitions", "4000") \
    .config("spark.executor.memoryOverhead", "4g") \
    .config("spark.sql.files.ignoreCorruptFiles", "true") \
    .config("spark.sql.parquet.enableVectorizedReader", "false") \
    .config("spark.sql.parquet.mergeSchema", "true") \
    .getOrCreate()

bash: /opt/conda/miniconda3/lib/libtinfo.so.6: no version information available (required by bash)
bash: /opt/conda/miniconda3/lib/libtinfo.so.6: no version information available (required by bash)
26/03/27 07:48:27 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/27 07:48:27 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/27 07:48:27 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Pleas

In [9]:
class RecommendationDataPipeline:
    def __init__(self, spark, config):
        self.spark = spark
        self.config = config
        self.train_df = None
        self.test_df = None
        self.metadata_df = None
        self.indexed_train = None
        self.indexed_test = None
        
    def _get_valid_paths(self, base_path, start_days_ago, end_days_ago):
        base_date = datetime.strptime(self.config['date'], "%Y-%m-%d")
        paths = []
        for i in range(start_days_ago, end_days_ago):
            target_date = (base_date - timedelta(days=i)).strftime("%Y-%m-%d")
            paths.append(f"{base_path}/day={target_date}")
        return paths

    def _read_and_union_paths(self, paths):
        df = None
        for path in paths:
            try:
                temp_df = self.spark.read.parquet(path)
                df = temp_df if df is None else df.unionByName(temp_df)
            except Exception:
                pass # Skipping missing paths
        return df

    def load_raw_data(self):
        print("Loading raw interactions and metadata...")
        test_paths = self._get_valid_paths(self.config['watch_history_path'], 0, self.config['test_days'])
        train_paths = self._get_valid_paths(self.config['watch_history_path'], self.config['test_days'], self.config['test_days'] + self.config['train_days'])
        
        self.test_df = self._read_and_union_paths(test_paths)
        self.train_df = self._read_and_union_paths(train_paths)
        
        # Load Metadata
        tv_df = self.spark.read.parquet(f"{self.config['db_path']}{self.config['date']}/enriched_tv.parquet")
        movie_df = self.spark.read.parquet(f"{self.config['db_path']}{self.config['date']}/enriched_movie.parquet")
        
        def clean_meta(df):
            return (df.filter((F.col('XstreamContentIds') != F.array()) & (F.col("published") == True))
                    .withColumn("item_id_exploded", F.explode("XstreamContentIds")) # Step 1: Explode
                    .select(
                        F.col("item_id_exploded").cast("string").alias("item_id"), # Step 2: Cast
                        "title", 
                        F.col('OriginalLanguage').alias('original_language').cast("string"),
                        "Genres"
                    ))
        self.metadata_df = clean_meta(tv_df).unionByName(clean_meta(movie_df)).distinct()
        

    def process_and_index_data(self):
        print("Filtering users, aggregating playtime, and building indexes...")
        
        # 1. Overlapping Users & Aggregation
        common_users = self.train_df.select("userId").distinct().join(self.test_df.select("userId").distinct(), "userId")
        
        train_filtered = self.train_df.join(common_users, "userId")
        test_filtered = self.test_df.join(common_users, "userId")
        
        train_stats = train_filtered.groupBy("userId", "item_id") \
            .agg(F.sum("total_play_time_sec").alias("total_playtime_combined")) \
            .withColumn("distinct_content_count", F.count("item_id").over(Window.partitionBy("userId"))) \
            .filter(F.col("total_playtime_combined").isNotNull() & ~F.isnan("total_playtime_combined"))
            
        # --- NEW 95TH PERCENTILE LOGIC ---
        # Get one row per user to ensure the percentile isn't skewed by row inflation
        user_content_counts = train_stats.select("userId", "distinct_content_count").distinct()
        
        # Calculate the 95th percentile
        p95_threshold = user_content_counts.stat.approxQuantile("distinct_content_count", [0.95], 0.001)[0]
        print(f"95th Percentile Threshold for max items watched: {p95_threshold}")
        
        # Apply both the minimum (>= 10) and the maximum (<= p95) filters
        als_input_base = train_stats.filter(
            (F.col("distinct_content_count") >= 10) & 
            (F.col("distinct_content_count") <= p95_threshold)
        )
        # ---------------------------------
        
        # Re-filter test users based on the valid base
        valid_users = als_input_base.select("userId").distinct()
        test_filtered = test_filtered.join(valid_users, "userId")
        
        # 2. Build Lookup Tables
        distinct_users = valid_users.rdd.zipWithIndex().toDF(["user_struct", "userIndex"]).select(F.col("user_struct.*"), F.col("userIndex").cast("int"))
        distinct_items = als_input_base.select("item_id").distinct().rdd.zipWithIndex().toDF(["item_struct", "itemIndex"]).select(F.col("item_struct.*"), F.col("itemIndex").cast("int"))
        
        # Break Lineage!
        distinct_users.write.mode("overwrite").parquet(f"{self.config['temp_path']}/user_lookup")
        distinct_items.write.mode("overwrite").parquet(f"{self.config['temp_path']}/item_lookup")
        
        user_lookup = self.spark.read.parquet(f"{self.config['temp_path']}/user_lookup")
        self.item_lookup = self.spark.read.parquet(f"{self.config['temp_path']}/item_lookup")

        # 3. Apply Indexes
        self.indexed_train = als_input_base.join(user_lookup, "userId").join(self.item_lookup, "item_id") \
            .withColumn("playtime_logged", F.log1p("total_playtime_combined")) \
            .select("userIndex", "itemIndex", "playtime_logged").repartition(1000).cache()
            
        self.indexed_test = test_filtered.join(user_lookup, "userId").join(self.item_lookup, "item_id") \
            .withColumn("playtime_logged", F.log1p("total_play_time_sec")) \
            .select("userIndex", "itemIndex", "playtime_logged").cache()

    def get_als_data(self):
        """Returns DataFrames optimized for Spark ALS."""
        ground_truth = self.indexed_test.groupBy("userIndex").agg(F.collect_set("itemIndex").alias("actual_items"))
        return self.indexed_train, ground_truth

    def get_lightfm_data(self, sample_fraction=0.05):
        """Samples, extracts features, and builds Scipy matrices for LightFM."""
        print(f"Preparing LightFM Matrices (Sampling {sample_fraction * 100}% of users)...")
        
        # Scrub test data of seen items
        clean_test = self.indexed_test.join(self.indexed_train.select("userIndex", "itemIndex"), on=["userIndex", "itemIndex"], how="left_anti")
        
        # Sample users to fit in driver memory
        sampled_users = self.indexed_train.select("userIndex").distinct().sample(withReplacement=False, fraction=sample_fraction, seed=42)
        
        train_pdf = self.indexed_train.join(sampled_users, "userIndex").toPandas()
        test_pdf = clean_test.join(sampled_users, "userIndex").toPandas()
        
        items_pdf = self.metadata_df.join(self.item_lookup, "item_id").select("itemIndex", "original_language", "Genres").dropDuplicates(["itemIndex"]).toPandas()
        
        # Clean & extract metadata features
        items_pdf['Genres'] = items_pdf['Genres'].fillna('Unknown').astype(str)
        items_pdf['original_language'] = items_pdf['original_language'].fillna('Unknown').astype(str)
        
        valid_items = np.unique(np.concatenate((train_pdf['itemIndex'], test_pdf['itemIndex'])))
        items_pdf = items_pdf[items_pdf['itemIndex'].isin(valid_items)]
        
        def extract_features(lang, genre_data):
            feats = [str(lang)]
            if genre_data.startswith('['):
                try: genre_data = ast.literal_eval(genre_data)
                except: genre_data = genre_data.replace('[', '').replace(']', '').replace("'", "").replace('"', "").split(',')
            else:
                genre_data = genre_data.split(',')
            feats.extend([g.strip() for g in genre_data])
            return list(set(feats))
            
        items_pdf['clean_features'] = items_pdf.apply(lambda row: extract_features(row['original_language'], row['Genres']), axis=1)
        all_features = list(set([feat for sublist in items_pdf['clean_features'] for feat in sublist]))
        
        # Build LightFM Dataset
        dataset = Dataset()
        all_users = np.unique(np.concatenate((train_pdf['userIndex'], test_pdf['userIndex'])))
        dataset.fit(users=all_users, items=valid_items, item_features=all_features)
        
        train_interact, train_weights = dataset.build_interactions(zip(train_pdf['userIndex'], train_pdf['itemIndex'], train_pdf['playtime_logged']))
        test_interact, _ = dataset.build_interactions(zip(test_pdf['userIndex'], test_pdf['itemIndex'], test_pdf['playtime_logged']))
        item_features = dataset.build_item_features((idx, feats) for idx, feats in zip(items_pdf['itemIndex'], items_pdf['clean_features']))
        
        return train_interact, train_weights, test_interact, item_features

In [10]:
config = {
    "date": "2026-03-22",
    "train_days": 175,
    "test_days": 30,
    "watch_history_path": "gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new",
    "db_path": "gs://wynk-ml-workspace/projects/xstream_nlu/catalog-db/",
    "temp_path": "gs://wynk-ml-workspace/_temp/harshith/als"
}

pipeline = RecommendationDataPipeline(spark, config)
pipeline.load_raw_data()
pipeline.process_and_index_data()

Loading raw interactions and metadata...


Filtering users, aggregating playtime, and building indexes...


95th Percentile Threshold for max items watched: 64.0


26/03/27 10:31:28 WARN DAGScheduler: Broadcasting large task binary with size 1111.8 KiB
26/03/27 10:31:35 WARN DAGScheduler: Broadcasting large task binary with size 1060.0 KiB


In [ ]:
# def tune_als(train_data, ground_truth, param_grid):
#     print("Starting ALS Hyperparameter Tuning...")
#     keys, values = zip(*param_grid.items())
#     combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    
#     for params in combinations:
            
#         # als = ALS(
#         #     userCol="userIndex", itemCol="itemIndex", ratingCol="playtime_logged",
#         #     implicitPrefs=True, maxIter=10, coldStartStrategy="drop",
#         #     rank=params['rank'], regParam=params['regParam'], alpha=params['alpha']
#         # )

#         als = ALS(
#             userCol="userIndex", itemCol="itemIndex", ratingCol="playtime_logged",
#             implicitPrefs=True, maxIter=10, coldStartStrategy="drop",
#             rank=params['rank'], alpha=params['alpha']
#         )
        
#         model = als.fit(train_data)
        
#         # Evaluate
#         user_recs = model.recommendForAllUsers(15) # Get top 30 recommendations for evaluation
#         user_recs_flat = user_recs.select("userIndex", F.col("recommendations.itemIndex").alias("predicted_items"))
#         eval_data = user_recs_flat.join(ground_truth, "userIndex").select("predicted_items", "actual_items").rdd.map(tuple)
        
#         metrics = RankingMetrics(eval_data)
#         map_metric = metrics.meanAveragePrecision
#         precision_15 = metrics.precisionAt(15)
#         recall_15 = metrics.recallAt(15)

#         print(f"ALS Params: {params} -> MAP: {map_metric:.4f}, R@15: {recall_15:.4f}")

# # 1. Fetch data formats
# als_train, als_ground_truth = pipeline.get_als_data()

# # 2. Define Grids
# als_param_grid = {
#     "rank": [50, 100],
#     "alpha": [1.0, 5.0]
# }

# tune_als(als_train, als_ground_truth, als_param_grid)

# print("Tuning complete! View results on your MLflow UI.")

Starting ALS Hyperparameter Tuning...


26/03/27 10:31:40 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/27 10:31:41 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/27 10:31:48 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/27 10:31:48 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/27 10:

ALS Params: {'rank': 50, 'alpha': 1.0} -> MAP: 0.1056, R@15: 0.2571


26/03/27 10:35:16 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 80 for reason Executor for container container_1764236692086_5808_01_000135 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/27 10:35:43 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 132 for reason Executor for container container_1764236692086_5808_01_000193 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/27 10:37:39 WARN DAGScheduler: Broadcasting large task binary with size 1174.2 KiB
26/03/27 10:37:43 WARN DAGScheduler: Broadcasting large task binary with size 1179.0 KiB
26/03/27 10:37:44 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/27 10:38:10 

ALS Params: {'rank': 50, 'alpha': 5.0} -> MAP: 0.0887, R@15: 0.2430


ERROR:root:KeyboardInterrupt while sending command.                (3 + 7) / 10]
Traceback (most recent call last):
  File "/opt/conda/default/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/default/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/default/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
import itertools
import pyspark.sql.functions as F
from pyspark.mllib.evaluation import RankingMetrics
from pyspark.ml.recommendation import ALS

# --- 1. NEW: The Popularity Penalty Function ---
def apply_popularity_penalty(als_train):
    print("Applying Popularity Penalty (TF-IDF) to training data...")
    
    # 1. Get total number of unique users (N)
    total_users = als_train.select("userIndex").distinct().count()
    
    # 2. Calculate how many unique users watched each item (U_i)
    item_frequencies = als_train.groupBy("itemIndex").agg(
        F.countDistinct("userIndex").alias("user_count")
    )
    
    # 3. Calculate the IDF weight (The Penalty)
    item_idf = item_frequencies.withColumn(
        "idf_weight",
        F.log((F.lit(total_users) + 1) / (F.col("user_count") + 1))
    )
    
    # 4. Join the penalty weights back to the main training data
    train_penalized = als_train.join(item_idf, on="itemIndex", how="left")
    
    # 5. Calculate the final penalized confidence score
    # Note: playtime_logged is already log-scaled in your pipeline, so we just multiply
    train_final = train_penalized.withColumn(
        "penalized_playtime",
        F.col("playtime_logged") * F.col("idf_weight")
    )
    
    print("Penalty applied successfully.")
    return train_final.select("userIndex", "itemIndex", "penalized_playtime")


# --- 2. UPDATED: Diagnostic Function (Now with Titles) ---
def check_popularity_bias(user_recs_flat, total_catalog_size, title_lookup_df):
    print("\n--- Running Popularity & Coverage Diagnostics ---")
    
    exploded_recs = user_recs_flat.select(
        "userIndex", 
        F.explode("predicted_items").alias("itemIndex")
    )
    
    total_recs_made = exploded_recs.count()
    
    unique_items_recommended = exploded_recs.select("itemIndex").distinct().count()
    coverage_percent = (unique_items_recommended / total_catalog_size) * 100
    
    print(f"Total Unique Items Recommended: {unique_items_recommended} out of {total_catalog_size}")
    print(f"Catalog Coverage: {coverage_percent:.2f}%")
    
    item_frequencies = exploded_recs.groupBy("itemIndex") \
                                    .count() \
                                    .orderBy(F.desc("count"))
    
    print("\nTop 10 Most Recommended Items (and how many users got them):")
    # Join with title lookup to show the actual movie/TV show names
    item_frequencies_with_titles = item_frequencies.join(title_lookup_df, "itemIndex", "left") \
                                                   .select("title", "count", "itemIndex") \
                                                   .orderBy(F.desc("count"))
    item_frequencies_with_titles.show(10, truncate=False)
    
    top_10_recs_count = item_frequencies.limit(10).agg(F.sum("count")).collect()[0][0]
    domination_percent = (top_10_recs_count / total_recs_made) * 100
    
    print(f"Blockbuster Domination: The Top 10 items make up {domination_percent:.2f}% of ALL recommendations made by the model.")


# --- 3. UPDATED: ALS Tuning Function ---
def tune_als(train_data, ground_truth, param_grid, total_catalog_size, title_lookup_df):
    print("\nStarting ALS Hyperparameter Tuning with Cohort-Based Recall...")
    keys, values = zip(*param_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    
    for params in combinations:
        print(f"\n--- Testing ALS Params: {params} ---")
        
        als = ALS(
            userCol="userIndex", 
            itemCol="itemIndex", 
            ratingCol="penalized_playtime", # <--- NOW USING PENALIZED SCORE
            implicitPrefs=True, 
            maxIter=10, 
            coldStartStrategy="drop",
            rank=params['rank'], 
            alpha=params['alpha']
        )
        
        model = als.fit(train_data)
        
        # Get top 30 recommendations
        user_recs = model.recommendForAllUsers(30) 
        user_recs_flat = user_recs.select(
            "userIndex", 
            F.col("recommendations.itemIndex").alias("predicted_items")
        )
        
        # Run diagnostics with titles
        check_popularity_bias(user_recs_flat, total_catalog_size, title_lookup_df)
        
        # Join predictions with ground truth
        joined_data = user_recs_flat.join(ground_truth, "userIndex")
        
        eval_df = joined_data.withColumn("actual_count", F.size(F.col("actual_items")))
        
        cohorts = {
            "Cohort A (1-3 items)": eval_df.filter((F.col("actual_count") >= 1) & (F.col("actual_count") <= 3)),
            "Cohort B (4-10 items)": eval_df.filter((F.col("actual_count") >= 4) & (F.col("actual_count") <= 10)),
            "Cohort C (>10 items)": eval_df.filter(F.col("actual_count") > 10),
            "Global (All Users)": eval_df
        }
        
        for cohort_name, cohort_df in cohorts.items():
            if cohort_df.head(1): 
                rdd_data = cohort_df.select("predicted_items", "actual_items").rdd.map(tuple)
                metrics = RankingMetrics(rdd_data)
                
                map_metric = metrics.meanAveragePrecision
                recall_30 = metrics.recallAt(30)
                user_count = cohort_df.count()
                
                print(f"[{cohort_name}] Users: {user_count} | MAP: {map_metric:.4f} | R@30: {recall_30:.4f}")
            else:
                print(f"[{cohort_name}] No users found in this cohort.")


# --- 4. NEW: Execution Block ---

# 1. Fetch data formats
als_train, als_ground_truth = pipeline.get_als_data()

# 2. Build Title Lookup DataFrame for Diagnostics
# Joining metadata with item_lookup to map itemIndex to title
title_lookup_df = pipeline.metadata_df.join(pipeline.item_lookup, "item_id") \
                                      .select("itemIndex", "title").distinct()

# 3. Calculate Catalog Size
total_catalog_size = als_train.select("itemIndex").distinct().count()
print(f"Total Catalog Size: {total_catalog_size}")

# 4. Apply Popularity Penalty to Training Data
als_train_penalized = apply_popularity_penalty(als_train)

# 5. Define Grids and Run
als_param_grid = {
    "rank": [50],
    "alpha": [1.0]
}

tune_als(als_train_penalized, als_ground_truth, als_param_grid, total_catalog_size, title_lookup_df)

Total Catalog Size: 35983
Applying Popularity Penalty (TF-IDF) to training data...
Penalty applied successfully.

Starting ALS Hyperparameter Tuning with Cohort-Based Recall...

--- Testing ALS Params: {'rank': 50, 'alpha': 1.0} ---


26/03/27 11:42:09 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/27 11:42:10 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.



--- Running Popularity & Coverage Diagnostics ---


26/03/27 11:44:51 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 143 for reason Container marked as failed: container_1764236692086_5808_01_000204 on host: dprc-wynk-prd-data-ds-machine-learning-common-sw-db0m.c.prj-wynk-prd-ds-svc-01.internal. Exit status: -100. Diagnostics: Container released on a *lost* node.
26/03/27 11:44:51 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 112 for reason Container marked as failed: container_1764236692086_5808_01_000169 on host: dprc-wynk-prd-data-ds-machine-learning-common-sw-db0m.c.prj-wynk-prd-ds-svc-01.internal. Exit status: -100. Diagnostics: Container released on a *lost* node.
26/03/27 11:44:51 ERROR YarnScheduler: Lost executor 143 on dprc-wynk-prd-data-ds-machine-learning-common-sw-db0m.c.prj-wynk-prd-ds-svc-01.internal: Container marked as failed: container_1764236692086_5808_01_000204 on host: dprc-wynk-prd-data-ds-machine-learning-common-sw-db0m.c.prj-wynk-prd-

Total Unique Items Recommended: 3278 out of 35983
Catalog Coverage: 9.11%

Top 10 Most Recommended Items (and how many users got them):


...

26/03/27 11:51:43 WARN DAGScheduler: Broadcasting large task binary with size 1174.2 KiB
26/03/27 11:51:50 WARN DAGScheduler: Broadcasting large task binary with size 1182.3 KiB
26/03/27 11:51:51 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/27 11:52:22 WARN DAGScheduler: Broadcasting large task binary with size 1174.2 KiB
26/03/27 11:52:29 WARN DAGScheduler: Broadcasting large task binary with size 1182.3 KiB
26/03/27 11:52:30 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/27 11:53:03 WARN DAGScheduler: Broadcasting large task binary with size 1174.2 KiB
26/03/27 11:53:10 WARN DAGScheduler: Broadcasting large task 

[Cohort C (>10 items)] Users: 236129 | MAP: 0.1236 | R@30: 0.2931


26/03/27 11:53:43 WARN DAGScheduler: Broadcasting large task binary with size 1174.2 KiB
26/03/27 11:53:50 WARN DAGScheduler: Broadcasting large task binary with size 1179.0 KiB
26/03/27 11:53:51 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/27 11:54:24 WARN DAGScheduler: Broadcasting large task binary with size 1174.2 KiB
26/03/27 11:54:31 WARN DAGScheduler: Broadcasting large task binary with size 1179.0 KiB
26/03/27 11:54:32 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/27 11:55:07 WARN DAGScheduler: Broadcasting large task binary with size 1181.2 KiB
26/03/27 11:55:14 WARN DAGScheduler: Broadcasting large task 

[Global (All Users)] Users: 1656482 | MAP: 0.1020 | R@30: 0.3534


26/03/27 12:01:14 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 150 for reason Executor for container container_1764236692086_5808_01_000211 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/27 12:01:14 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 161 for reason Executor for container container_1764236692086_5808_01_000222 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/27 12:01:14 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 71 for reason Executor for container container_1764236692086_5808_01_000125 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/27 12:01:14 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 67 for reason Executor for container container_1764236692086